In [1]:
import os
os.chdir(r"D:\Study\Programs\trading")

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import ast
from strategies_dev.utils import flatten_depth

In [3]:
date_ = "30APR2026"
file_name = "NIFTY2650523950CE"


In [4]:
def flatten_depth2(row):
    depth = row['depth']
    data = {}

    for side, prefix in [('buy', 'bid'), ('sell', 'ask')]:
        for i in range(5):
            # Use .get() or list indexing safely
            entry = depth[side][i] if i < len(depth[side]) else {}
            data[f'{prefix}_price_{i+1}'] = entry.get('price')
            data[f'{prefix}_size_{i+1}'] = entry.get('quantity')
    return pd.Series(data)

def update_volume(row):
    if row['volume_traded_actual'] == row['volume_traded_prev']:
        return np.nan
    else:
        return row['volume_traded_actual'] - row['volume_traded_prev']

In [5]:
file_path = Path(fr"D:\Study\Programs\trading\assets\logs\{date_}\extracted_symbols\{file_name}.xlsx")
required_cols = ["last_trade_time", "last_price", "last_traded_quantity", "volume_traded", "depth"]
df = pd.read_excel(file_path)[required_cols]
df["depth"] = df.apply(lambda row: ast.literal_eval(row["depth"]), axis=1)

df = df.join(df.apply(flatten_depth2, axis=1, result_type='expand'))
df["volume_traded_actual"] = df["volume_traded"]
df["volume_traded_prev"] = df["volume_traded_actual"].shift(1)

df["volume_traded"] = df.apply(lambda row: update_volume(row), axis=1)
df['volume_traded'] = df['volume_traded'].ffill()

df = df.drop(columns=['depth', 'volume_traded_actual', 'volume_traded_prev'])

In [6]:
df.columns

Index(['last_trade_time', 'last_price', 'last_traded_quantity',
       'volume_traded', 'bid_price_1', 'bid_size_1', 'bid_price_2',
       'bid_size_2', 'bid_price_3', 'bid_size_3', 'bid_price_4', 'bid_size_4',
       'bid_price_5', 'bid_size_5', 'ask_price_1', 'ask_size_1', 'ask_price_2',
       'ask_size_2', 'ask_price_3', 'ask_size_3', 'ask_price_4', 'ask_size_4',
       'ask_price_5', 'ask_size_5'],
      dtype='str')

In [7]:
df.to_csv(r"D:\Study\Programs\trading\Experiments\TCNN\data\data.csv", index=False)

In [55]:
# Step 1: Pre-clean (must do first)

df = df.sort_values('last_trade_time').reset_index(drop=True)

# convert time if not already
df['last_trade_time'] = pd.to_datetime(df['last_trade_time'])

# time delta (seconds)
df['dt'] = df['last_trade_time'].diff().dt.total_seconds().fillna(0)

In [57]:
### Step 2: Trade-based features (pressure + momentum)

In [58]:
# 2.1 Price return (very important)

df['price_return'] = df['last_price'].pct_change().fillna(0)

In [59]:
df['price_velocity'] = df['last_price'].diff().fillna(0)

In [60]:
df['trade_size'] = df['last_traded_quantity']

In [61]:
df['volume_change'] = df['volume_traded'].diff().fillna(0)

In [62]:
df['mid_price'] = (df['bid_price_1'] + df['ask_price_1']) / 2

In [63]:
df['spread'] = df['ask_price_1'] - df['bid_price_1']

In [64]:
bid_sizes = [f'bid_size_{i}' for i in range(1, 6)]
ask_sizes = [f'ask_size_{i}' for i in range(1, 6)]

df['total_bid_size'] = df[bid_sizes].sum(axis=1)
df['total_ask_size'] = df[ask_sizes].sum(axis=1)

df['orderbook_imbalance'] = (
    (df['total_bid_size'] - df['total_ask_size']) /
    (df['total_bid_size'] + df['total_ask_size'] + 1e-9)
)

In [65]:
df['l1_imbalance'] = (
    (df['bid_size_1'] - df['ask_size_1']) /
    (df['bid_size_1'] + df['ask_size_1'] + 1e-9)
)

In [66]:
df['trade_to_mid'] = df['last_price'] - df['mid_price']

In [67]:
df['is_buy'] = (df['last_price'] >= df['ask_price_1']).astype(int)
df['is_sell'] = (df['last_price'] <= df['bid_price_1']).astype(int)

In [68]:
df['tick_speed'] = 1 / (df['dt'] + 1e-6)

In [69]:
df['activity_10'] = df['dt'].rolling(10).mean().fillna(0)

In [70]:
df

,last_trade_time,last_price,last_traded_quantity,volume_traded,bid_price_1,bid_size_1,bid_price_2,bid_size_2,bid_price_3,bid_size_3,...,spread,total_bid_size,total_ask_size,orderbook_imbalance,l1_imbalance,trade_to_mid,is_buy,is_sell,tick_speed,activity_10
0,2026-04-29 15:29:59,340.30,65,NaN,0.00,0.0,0.00,0.0,0.00,0.0,...,0.00,0.0,0.0,0.000000,0.000000,340.300,1,0,1000000.000000,0.0
1,2026-04-30 09:15:00,216.70,65,1755.0,197.50,325.0,197.45,130.0,195.75,260.0,...,19.20,1170.0,715.0,0.241379,0.666667,9.600,1,0,0.000016,0.0
2,2026-04-30 09:15:00,211.00,130,1755.0,197.50,325.0,197.45,130.0,195.75,260.0,...,19.20,1170.0,715.0,0.241379,0.666667,3.900,0,0,1000000.000000,0.0
3,2026-04-30 09:15:01,197.70,65,14300.0,195.55,260.0,195.50,65.0,195.45,260.0,...,2.10,975.0,780.0,0.111111,-0.111111,1.100,1,0,0.999999,0.0
4,2026-04-30 09:15:01,197.45,65,14300.0,195.55,260.0,195.50,65.0,195.45,260.0,...,2.10,975.0,780.0,0.111111,-0.111111,0.850,0,0,1000000.000000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31522,2026-04-30 15:29:57,251.35,195,975.0,251.35,520.0,251.30,390.0,250.00,65.0,...,1.95,1235.0,1105.0,0.055556,0.454545,-0.975,0,1,1000000.000000,0.7
31523,2026-04-30 15:29:59,250.60,65,715.0,252.00,325.0,251.60,195.0,250.65,520.0,...,0.70,1560.0,910.0,0.263158,0.250000,-1.750,0,1,0.500000,0.8
31524,2026-04-30 15:29:59,252.00,65,715.0,252.00,325.0,251.60,195.0,250.65,520.0,...,0.70,1560.0,910.0,0.263158,0.250000,-0.350,0,1,1000000.000000,0.8
31525,2026-04-30 15:29:59,250.60,65,715.0,252.00,325.0,251.60,195.0,250.65,520.0,...,0.70,1560.0,910.0,0.263158,0.250000,-1.750,0,1,1000000.000000,0.8


### Labelling

In [71]:
import numpy as np

# convert to numpy for speed
times = df['last_trade_time'].values
prices = df['last_price'].values

future_price = np.full(len(df), np.nan)

j = 0
for i in range(len(df)):
    target_time = times[i] + np.timedelta64(15, 's')

    while j < len(df) and times[j] < target_time:
        j += 1

    if j < len(df):
        future_price[i] = prices[j]

df['future_price'] = future_price

In [72]:
df['future_return'] = (df['future_price'] - df['last_price']) / df['last_price']

In [73]:
threshold = 0.0005  # 0.05% (tune this)

In [74]:
def label_fn(x):
    if x > threshold:
        return 1   # UP
    elif x < -threshold:
        return -1  # DOWN
    else:
        return 0   # NO MOVE

df['target'] = df['future_return'].apply(label_fn)

In [75]:
df = df.dropna(subset=['future_price'])

In [76]:
print(df['target'].value_counts(normalize=True))

target
 1    0.484161
-1    0.471210
 0    0.044629
Name: proportion, dtype: float64


### Sequence Modelling

In [78]:
SEQ_LEN = 60   # ~30–60 seconds of context

features = [
    'price_return',
    'price_velocity',
    'trade_size',
    'volume_change',
    'spread',
    'mid_price',
    'orderbook_imbalance',
    'l1_imbalance',
    'trade_to_mid',
    'is_buy',
    'is_sell',
    'tick_speed',
    'activity_10'
]

In [79]:
import numpy as np

data = df[features].values
targets = df['target'].values

In [80]:
X = []
y = []

for i in range(SEQ_LEN, len(df)):
    X.append(data[i-SEQ_LEN:i])
    y.append(targets[i])

X = np.array(X)
y = np.array(y)

In [81]:
print(X.shape)
print(y.shape)

(31444, 60, 13)
(31444,)


In [82]:
split = int(0.8 * len(X))

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

In [83]:
mean = X_train.mean(axis=(0,1))
std = X_train.std(axis=(0,1)) + 1e-9

X_train = (X_train - mean) / std
X_test = (X_test - mean) / std

In [84]:
y_train = (y_train + 1)  # [-1,0,1] → [0,1,2]
y_test  = (y_test + 1)